# Repeated ROST runs and model statistics

This notebook is a generic example for evaluating stochastic `rostpy` topic models.

ROST refinement is stochastic, so the same input data and same model settings can produce different results across runs. A practical workflow is to run the model multiple times, compute model statistics for each run, and keep the best run according to a transparent criterion.

This example uses a small synthetic abundance table and focuses only on model statistics. It does not include domain-specific visualization or analysis.

In [ ]:
import itertools

import numpy as np
import pandas as pd
import rostpy

## Example abundance table

Rows are samples. Columns are words, taxa, or other count-like features. Values are abundance-like measurements.

In [ ]:
abundance = pd.DataFrame(
    {
        "word_A": [2500, 2200, 1800, 100, 50, 100],
        "word_B": [1800, 1500, 1700, 200, 100, 150],
        "word_C": [400, 300, 500, 600, 500, 400],
        "word_D": [100, 100, 200, 1800, 2200, 2000],
        "word_E": [50, 100, 150, 1600, 1900, 2100],
    },
    index=[f"sample_{i:03d}" for i in range(1, 7)],
)

words = abundance.columns.to_list()
sample_ids = abundance.index.to_list()

abundance

Expected output: a `6 x 5` table with six samples and five words/features.

## Deterministic abundance scaling

ROST expects repeated integer word IDs. This example converts abundance values to deterministic scaled integer counts. This preserves absolute abundance differences while keeping the token lists small.

In [ ]:
scale = 100

# round() avoids systematically flooring near-threshold values to zero.
counts = abundance.div(scale).round().astype(int)
token_counts = counts.sum(axis=1)

print("count table shape:", counts.shape)
print("token counts per sample:")
print(token_counts)
print("empty samples:", int((token_counts == 0).sum()))

counts

Expected output: `counts` is the integer table used to build ROST observations. `token_counts` reports how many word tokens each sample contributes.

## Helper functions

In [ ]:
def make_observations(counts: pd.DataFrame) -> dict[int, list[int]]:
    """Convert each count row to repeated integer word IDs."""
    observations = {}
    for t, (_, row) in enumerate(counts.iterrows()):
        obs = []
        for word_id, count in enumerate(row):
            obs.extend([word_id] * int(count))
        if obs:
            observations[t] = obs
    return observations


def fit_rost_once(counts, K, n_epochs, nt=1, alpha=0.5, beta=0.5, gamma=0.1):
    """Fit one stochastic ROST_t model from an integer count table."""
    observations = make_observations(counts)
    model = rostpy.ROST_t(V=counts.shape[1], K=K, alpha=alpha, beta=beta, gamma=gamma)

    for t, obs in observations.items():
        model.add_observation([t], obs)

    for _ in range(n_epochs):
        # Use positional arguments for compatibility with existing bindings.
        rostpy.parallel_refine(model, nt)

    return model, observations


def sample_topic_probabilities(model, observations, sample_ids):
    """Convert word-level topic labels to sample-topic probabilities."""
    topic_counts = pd.DataFrame(0, index=sample_ids, columns=[f"topic_{k + 1}" for k in range(model.K)])
    for t in observations:
        sample_id = sample_ids[t]
        for topic in model.get_ml_topics_for_pose([t]):
            topic_counts.loc[sample_id, f"topic_{topic + 1}"] += 1
    return topic_counts.div(topic_counts.sum(axis=1), axis=0).fillna(0)


def topic_word_probabilities(model, words):
    """Normalize the topic-word count matrix into probabilities."""
    topic_word_counts = pd.DataFrame(
        model.get_topic_model(),
        index=[f"topic_{k + 1}" for k in range(model.K)],
        columns=words,
    )
    return topic_word_counts.div(topic_word_counts.sum(axis=1), axis=0).fillna(0)


def topic_entropy(topic_word_prob_df):
    """Compute Shannon entropy for each topic-word distribution."""
    p = topic_word_prob_df.to_numpy(dtype=float)
    values = -(p * np.log(p + 1e-12)).sum(axis=1)
    return pd.Series(values, index=topic_word_prob_df.index, name="entropy")


def mean_pairwise_topic_similarity(topic_word_prob_df):
    """Compute mean pairwise cosine similarity among topics."""
    x = topic_word_prob_df.to_numpy(dtype=float)
    if len(x) < 2:
        return np.nan
    similarities = []
    for i, j in itertools.combinations(range(len(x)), 2):
        denom = np.linalg.norm(x[i]) * np.linalg.norm(x[j])
        similarities.append(float(np.dot(x[i], x[j]) / denom) if denom else np.nan)
    return float(np.nanmean(similarities))


def umass_coherence(topic_word_prob_df, counts, top_n=5):
    """Compute a simple UMass-style coherence score for each topic."""
    presence = counts.gt(0)
    scores = {}
    for topic, probs in topic_word_prob_df.iterrows():
        top_words = probs.sort_values(ascending=False).head(top_n).index.to_list()
        terms = []
        for i in range(1, len(top_words)):
            for j in range(i):
                wi = top_words[i]
                wj = top_words[j]
                d_wi_wj = int((presence[wi] & presence[wj]).sum())
                d_wj = int(presence[wj].sum())
                terms.append(np.log((d_wi_wj + 1) / max(d_wj, 1)))
        scores[topic] = float(np.mean(terms)) if terms else np.nan
    return pd.Series(scores, name="umass_coherence")


def summarize_model(model, observations, counts, words, sample_ids):
    """Extract generic statistics from one fitted ROST model."""
    sample_topic_df = sample_topic_probabilities(model, observations, sample_ids)
    topic_word_df = topic_word_probabilities(model, words)
    entropy = topic_entropy(topic_word_df)
    coherence = umass_coherence(topic_word_df, counts)
    return {
        "perplexity": model.perplexity(),
        "num_cells": model.num_cells(),
        "refine_count": model.get_refine_count(),
        "word_refine_count": model.get_word_refine_count(),
        "mean_topic_entropy": float(entropy.mean()),
        "mean_umass_coherence": float(coherence.mean()),
        "mean_topic_similarity": mean_pairwise_topic_similarity(topic_word_df),
        "sample_topic_df": sample_topic_df,
        "topic_word_df": topic_word_df,
        "topic_entropy": entropy,
        "topic_coherence": coherence,
    }

## Fit one model and inspect outputs

In [ ]:
model, observations = fit_rost_once(counts, K=2, n_epochs=50, nt=1)
stats = summarize_model(model, observations, counts, words, sample_ids)

print("perplexity:", stats["perplexity"])
print("number of cells:", stats["num_cells"])
print("cell refinements:", stats["refine_count"])
print("word refinements:", stats["word_refine_count"])

In [ ]:
stats["sample_topic_df"]

In [ ]:
stats["topic_word_df"]

In [ ]:
pd.concat([stats["topic_entropy"], stats["topic_coherence"]], axis=1)

## Repeat stochastic runs and select the best run

The selection score combines lower perplexity, higher coherence, and lower topic similarity. You can edit this criterion for your own analysis.

In [ ]:
K = 2
n_runs = 10
n_epochs = 50

run_records = []
run_outputs = {}

for run_id in range(n_runs):
    model, observations = fit_rost_once(counts, K=K, n_epochs=n_epochs, nt=1)
    stats = summarize_model(model, observations, counts, words, sample_ids)
    run_records.append(
        {
            "run_id": run_id,
            "perplexity": stats["perplexity"],
            "mean_umass_coherence": stats["mean_umass_coherence"],
            "mean_topic_entropy": stats["mean_topic_entropy"],
            "mean_topic_similarity": stats["mean_topic_similarity"],
            "refine_count": stats["refine_count"],
            "word_refine_count": stats["word_refine_count"],
        }
    )
    run_outputs[run_id] = {"model": model, "observations": observations, "stats": stats}

run_summary_df = pd.DataFrame(run_records)
run_summary_df["perplexity_rank"] = run_summary_df["perplexity"].rank(method="min", ascending=True)
run_summary_df["coherence_rank"] = run_summary_df["mean_umass_coherence"].rank(method="min", ascending=False)
run_summary_df["similarity_rank"] = run_summary_df["mean_topic_similarity"].rank(method="min", ascending=True)
run_summary_df["selection_score"] = (
    run_summary_df["perplexity_rank"]
    + run_summary_df["coherence_rank"]
    + run_summary_df["similarity_rank"]
)
run_summary_df = run_summary_df.sort_values("selection_score").reset_index(drop=True)
run_summary_df

In [ ]:
best_run_id = int(run_summary_df.iloc[0]["run_id"])
best = run_outputs[best_run_id]

print("best run:", best_run_id)
print("best run perplexity:", best["stats"]["perplexity"])
print("best run mean coherence:", best["stats"]["mean_umass_coherence"])
print("best run mean topic similarity:", best["stats"]["mean_topic_similarity"])

## Inspect the selected run

In [ ]:
best["stats"]["sample_topic_df"]

In [ ]:
best["stats"]["topic_word_df"]

In [ ]:
pd.concat([best["stats"]["topic_entropy"], best["stats"]["topic_coherence"]], axis=1)